In [0]:
df = spark\
.read\
.option("inferSchema", "true")\
.option("header", "true")\
.csv("/databricks-datasets/flights/departuredelays.csv")

In [0]:
df.createOrReplaceTempView("flight_df")

In [0]:
%sql
select count(*) from flight_df

count(1)
1391578


In [0]:
df.printSchema()

root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)



In [0]:
df.count()

Out[5]: 1391578

In [0]:
%sql
select * from flight_df limit 10

date,delay,distance,origin,destination
1011245,6,602,ABE,ATL
1020600,-8,369,ABE,DTW
1021245,-2,602,ABE,ATL
1020605,-4,602,ABE,ATL
1031245,-4,602,ABE,ATL
1030605,0,602,ABE,ATL
1041243,10,602,ABE,ATL
1040605,28,602,ABE,ATL
1051245,88,602,ABE,ATL
1050605,9,602,ABE,ATL


In [0]:
%sql
select count(1),origin from flight_df group by origin order by 1 desc

count(1),origin
91484,ATL
68482,DFW
64228,ORD
54086,LAX
53148,DEN
43361,IAH
40155,PHX
39483,SFO
33107,LAS
28402,CLT


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select max(date),min(date) from flight_df

max(date),min(date)
3312359,1010005


In [0]:
%sql
select count(1), date  from flight_df group by date having count(1)> 1 order by 1 desc limit 100

count(1),date
368,3170600
368,3240600
367,3310600
367,3100600
366,3280600
365,3210600
365,3140600
363,3270600
363,3200600
363,3190600


In [0]:
%sql
select * from flight_df where date=3180600 limit 100

date,delay,distance,origin,destination
3180600,-4,494,ABQ,DFW
3180600,0,1103,ABQ,ATL
3180600,-3,285,ABQ,PHX
3180600,-3,423,ABQ,LAS
3180600,144,126,ABY,ATL
3180600,-4,78,ACT,DFW
3180600,-6,435,AEX,ATL
3180600,-8,247,AEX,DFW
3180600,-5,741,ALB,ATL
3180600,-3,251,ALB,BWI


In [0]:
%sql
select count(1),date,origin,destination from flight_df group by date,origin,destination having count(1) > 1 limit 100

count(1),date,origin,destination
2,1062030,ATL,DCA
2,1141055,ATL,MCO
2,1062055,DEN,LAX
2,1311915,DFW,SFO
2,1280700,HOU,ATL
2,1271730,IAH,SEA
2,1220935,BNA,ORD
2,1071030,EWR,ATL
2,1011515,ATL,PHL
2,1101445,ATL,LGA


In [0]:
%sql
select * from flight_df where date='1120815' and origin='DEN' and destination='PDX'

date,delay,distance,origin,destination
1120815,-9,861,DEN,PDX
1120815,19,861,DEN,PDX


In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) date_rank
 from flight_df a where a.date='1120815' and a.origin='DEN' and a.destination='PDX' 

date,delay,distance,origin,destination,flight_rank,date_rank
1120815,-9,861,DEN,PDX,1,1
1120815,19,861,DEN,PDX,2,1


In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) date_rank
 from flight_df a  order by date limit 100

date,delay,distance,origin,destination,flight_rank,date_rank
1010005,-8,2024,LAX,PBI,1,1
1010010,-6,1980,SEA,CLT,1,2
1010020,0,1273,SFO,DFW,1,3
1010020,-2,1995,SFO,CLT,2,3
1010023,14,1421,SFO,IAH,3,4
1010025,33,1198,LAX,IAH,2,5
1010025,-3,1452,PHX,DTW,1,5
1010029,49,1061,LAS,IAH,1,6
1010030,-2,1983,PDX,CLT,1,7
1010030,-8,1518,LAS,ATL,2,7


In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
date_add(to_timestamp('1999-12-31', 'yyyy-MM-dd'),dense_rank() over(partition by 1 order by date)) Date
 from flight_df a  order by date limit 10

date,delay,distance,origin,destination,flight_rank,Date
1010005,-8,2024,LAX,PBI,1,2000-01-01
1010010,-6,1980,SEA,CLT,1,2000-01-02
1010020,0,1273,SFO,DFW,1,2000-01-03
1010020,-2,1995,SFO,CLT,2,2000-01-03
1010023,14,1421,SFO,IAH,3,2000-01-04
1010025,33,1198,LAX,IAH,2,2000-01-05
1010025,-3,1452,PHX,DTW,1,2000-01-05
1010029,49,1061,LAS,IAH,1,2000-01-06
1010030,-7,2191,SFO,PHL,4,2000-01-07
1010030,-2,1983,PDX,CLT,1,2000-01-07


In [0]:
from pyspark.sql.functions import round

In [0]:
%sql
Create or replace temp view flight_Data_crafted as 
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) Day_rank,
date_add(to_timestamp('1999-12-31', 'yyyy-MM-dd'),int(round(dense_rank() over(partition by 1 order by date)/24))) new_date
 from flight_df a  order by date 

In [0]:
%sql
select * from flight_Data_crafted order by Day_rank desc limit 10

date,delay,distance,origin,destination,flight_rank,Day_rank,new_date
3312359,-3,1413,DEN,JFK,53146,95312,2010-11-14
3312359,-7,1405,JFK,PSE,23569,95312,2010-11-14
3312359,39,1859,SFO,ATL,39482,95312,2010-11-14
3312359,-1,2090,ANC,DEN,3632,95312,2010-11-14
3312359,-2,1586,ABQ,JFK,5739,95312,2010-11-14
3312359,-6,1369,JFK,BQN,23570,95312,2010-11-14
3312359,6,1604,SFO,ORD,39483,95312,2010-11-14
3312359,67,1388,JFK,SJU,23571,95312,2010-11-14
3312359,0,1691,LAX,ATL,54085,95312,2010-11-14
3312359,-7,1388,JFK,SJU,23572,95312,2010-11-14


In [0]:
from pyspark.sql.functions import window,col,column,desc
df_new =spark.sql("select new_date as date,delay,origin,destination from flight_Data_crafted")

In [0]:
df_new.printSchema()

root
 |-- date: date (nullable = true)
 |-- delay: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)



In [0]:
df_7_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_7_days.take(5))

origin,window,sum(delay)
AVL,"List(2000-01-06T00:00:00.000+0000, 2000-01-13T00:00:00.000+0000)",-1
AUS,"List(2000-01-13T00:00:00.000+0000, 2000-01-20T00:00:00.000+0000)",82
GUC,"List(2000-01-13T00:00:00.000+0000, 2000-01-20T00:00:00.000+0000)",107
LAX,"List(2000-01-27T00:00:00.000+0000, 2000-02-03T00:00:00.000+0000)",1310
JAX,"List(2000-01-27T00:00:00.000+0000, 2000-02-03T00:00:00.000+0000)",1060


In [0]:
df_30_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_30_days.sort("origin").take(5)

  File <command-2113719867016691>:5
    display(df_30_days.sort("origin").take(5)
                                             ^
SyntaxError: unexpected EOF while parsing


In [0]:
df_30_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_30_days.where("origin='LAX'").take(5))

origin,window,sum(delay)
LAX,"List(2000-01-27T00:00:00.000+0000, 2000-02-03T00:00:00.000+0000)",1310
LAX,"List(2001-12-06T00:00:00.000+0000, 2001-12-13T00:00:00.000+0000)",593
LAX,"List(2006-08-10T00:00:00.000+0000, 2006-08-17T00:00:00.000+0000)",19
LAX,"List(2010-06-17T00:00:00.000+0000, 2010-06-24T00:00:00.000+0000)",940
LAX,"List(2002-08-22T00:00:00.000+0000, 2002-08-29T00:00:00.000+0000)",906


In [0]:
display(df_new.take(5))

date,delay,origin,destination
1999-12-31,-8,LAX,PBI
1999-12-31,-6,SEA,CLT
1999-12-31,0,SFO,DFW
1999-12-31,-2,SFO,CLT
1999-12-31,14,SFO,IAH


In [0]:
%sql
select * from flight_Data_crafted limit 10

date,delay,distance,origin,destination,flight_rank,Day_rank,new_date
1010005,-8,2024,LAX,PBI,1,1,1999-12-31
1010010,-6,1980,SEA,CLT,1,2,1999-12-31
1010020,0,1273,SFO,DFW,1,3,1999-12-31
1010020,-2,1995,SFO,CLT,2,3,1999-12-31
1010023,14,1421,SFO,IAH,3,4,1999-12-31
1010025,33,1198,LAX,IAH,2,5,1999-12-31
1010025,-3,1452,PHX,DTW,1,5,1999-12-31
1010029,49,1061,LAS,IAH,1,6,1999-12-31
1010030,-7,2191,SFO,PHL,4,7,1999-12-31
1010030,-2,1983,PDX,CLT,1,7,1999-12-31
